<a href="https://colab.research.google.com/github/ahushka/fold_tree/blob/condacolab/notebooks/Foldtree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/DessimozLab/fold_tree/raw/main/foldtree_logo.png" height="200" align="right" style="height:240px">


##Foldtree - construct trees from protein structures

Easy to use notebook to construct phylogenetic trees from protein structure using [Foldtree](https://github.com/DessimozLab/fold_tree).
Foldtree is powered by [Foldseek](foldseek.com) to align protein structures and generate the distance matrix used for tree computation.

[Moi D., Bernard C., Steinegger M., Nevers Y., Langleib M., Dessimoz C.
Structural phylogenetics unravels the evolutionary diversification of communication systems in gram-positive bacteria and their viruses,
*biorxiv*, 2023](https://www.biorxiv.org/content/10.1101/2023.09.19.558401v2)

In [1]:
#@markdown ### Input (custom PDBs upload, identifier list, cluster ids)
from google.colab import files
import os
import re
import hashlib
import random
import zipfile

input_type = "afdb_cluster" #@param ["afdb_cluster", "identifier", "custom"]
#
#@markdown - afdb_cluster = identifier of an AFDB cluster,
#@markdown - identifier" = uniprot identifer (e.g. A0A074YNE0) list line by line,
#@markdown - custom - zip file with PDBs

cluster_id = "A0A2A4MYV0" #@param {type:"string"}
jobname = 'test' #@param {type:"string"}

def add_hash(x,y):
  return x+"_"+hashlib.sha1(y.encode()).hexdigest()[:5]

from sys import version_info
python_version = f"{version_info.major}.{version_info.minor}"


basejobname = "".join(jobname.split())
basejobname = re.sub(r'\W+', '', basejobname)
jobname = add_hash(basejobname, cluster_id)

# check if directory with jobname exists
def check(folder):
  if os.path.exists(folder):
    return False
  else:
    return True
if not check(jobname):
  n = 0
  while not check(f"{jobname}_{n}"): n += 1
  jobname = f"{jobname}_{n}"

# make directory to save results
os.makedirs(jobname, exist_ok=True)

if input_type == "custom":
  input_file = os.path.join(jobname,f"{jobname}.zip")
  if not os.path.isfile(input_file):
    zipfiles = files.upload()
    zipfile_name = list(zipfiles.keys())[0]
    os.rename(zipfile_name, input_file)
    # Unzipping the file
    with zipfile.ZipFile(input_file, 'r') as zip_ref:
      zip_ref.extractall(jobname)
    os.remove(input_file)

    input_file = os.path.join(jobname,f"identifiers.txt")
    with open(input_file, "w") as f:
      f.write("")
    os.mkdir(os.path.join(jobname,"structs"))
    for file in os.listdir(jobname):
      if file.endswith(".pdb"):
        os.rename(os.path.join(jobname,file), os.path.join(jobname,"structs",file))



elif input_type == "afdb_cluster":
  import requests
  # Define the endpoint and parameters
  base_url = "https://cluster.foldseek.com/api/cluster/"
  params = {
      "format": "accessions",
      "groupBy": "",
      "groupDesc": "",
      "itemsPerPage": 10,
      "multiSort": "false",
      "mustSort": "false",
      "page": 1,
      "sortBy": "",
      "sortDesc": "false"
  }

  # Make the request
  response = requests.get(f"{base_url}{cluster_id}/members", params=params)

  # Ensure the request was successful
  response.raise_for_status()

  # Save the response content to a file
  with open(f"{jobname}/identifiers.txt", "w") as file:
      file.write(response.text)
elif input_type == "identifier":
  input_file = os.path.join(jobname,f"identifiers.txt")
  if not os.path.isfile(input_file):
    identifierfiles = files.upload()
    identifierfilename = list(identifierfiles.keys())[0]
    os.rename(identifierfilename, input_file)



In [2]:
#installs mamba from https://pypi.org/project/condacolab/
!pip install -q condacolab
import condacolab
condacolab.install()
!mamba init

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/23.11.0-0/Mambaforge-23.11.0-0-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:12
🔁 Restarting kernel...
no change     /usr/local/condabin/conda
no change     /usr/local/bin/conda
no change     /usr/local/bin/conda-env
no change     /usr/local/bin/activate
no change     /usr/local/bin/deactivate
no change     /usr/local/etc/profile.d/conda.sh
no change     /usr/local/etc/fish/conf.d/conda.fish
no change     /usr/local/shell/condabin/Conda.psm1
no change     /usr/local/shell/condabin/conda-hook.ps1
no change     /usr/local/lib/python3.10/site-packages/xontrib/conda.xsh
no change     /usr/local/etc/profile.d/conda.csh
modified      /root/.bashrc

==> For changes to take effect, close and re-open your current shell. <==

Added mamba to /root/.bashrc

==> For changes to take effect, close and re-open your current shell. <==



In [1]:
#@title Install dependecies
%%bash -s $python_version
PYTHON_VERSION=$1
# Check if fold_tree directory exists and remove it
if [ -d "fold_tree" ]; then
  rm -r fold_tree
fi
pip install -q biopython ete3 pyqt5 wget statsmodels toytree toyplot requests tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 51.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.2/279.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.9/

In [5]:
!git clone -q https://github.com/DessimozLab/fold_tree
#!git clone -q https://github.com/DessimozLab/fold_tree --branch foldtreeserver


In [2]:
!mamba install -q -c bioconda -c conda-forge -c nodefaults python="${PYTHON_VERSION}" foldseek snakemake snakedeploy snakefmt iqtree muscle quicktree fasttree=2.1.11 clustalo=1.2.4 python-wget  #ete3 statsmodels toyplot tqdm requests #biopython toytree

Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done


In [24]:
!ls /content/fold_tree/workflow/

Astral		  benchmarking	fold_tree      MDbootstrap    server_seqtree
astral_benchmark  config	getstats       ML3ditree      tcoffee
benchmark_all	  FidentXaln	iqtreeXsingle  server_search


In [ ]:
!mamba env create -n foldtree -f /content/fold_tree/workflow/config/foldtree.yaml/
!mamba activate foldtree
!mamba init


[+] 0.0s
bioconda/linux-64 ..  ⣾  [+] 0.1s
bioconda/linux-64 ..  ⣾  bioconda/linux-64 (check zst)                     
[+] 0.0s
bioconda/noarch (check zst)                        Checked  0.1s
[+] 0.0s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  
bioconda/linux-64     ⣾  
bioconda/noarch       ⣾  [+] 0.1s
conda-forge/linux-64  ⣾  
conda-forge/noarch    ⣾  
bioconda/linux-64      1%
bioconda/noarch       ⣾  [+] 0.2s
conda-forge/linux-64   1%
conda-forge/noarch     2%
bioconda/linux-64      5%
bioconda/noarch        2%[+] 0.3s
conda-forge/linux-64   1%
conda-forge/noarch     5%
bioconda/linux-64      5%
bioconda/noarch       43%[+] 0.4s
conda-forge/linux-64   1%
conda-forge/noarch     5%
bioconda/linux-64     42%
bioconda/noarch       43%[+] 0.5s
conda-forge/linux-64   2%
conda-forge/noarch     5%
bioconda/linux-64     42%
bioconda/noarch       76%[+] 0.6s
conda-forge/linux-64   2%
conda-forge/noarch    14%
bioconda/linux-64     42%
bioconda/noarch       76%[+] 0.7s
conda-forge/li

In [13]:
#@title Run Foldtree
#%%bash -s $jobname $input_type
%%bash -s "$jobname" "$input_type"
JOBNAME=$1
INPUT_TYPE=$2
SUFFIX=""
if [[ $INPUT_TYPE = "custom" ]]; then
  mkdir -p "${JOBNAME}/structs"
  mv "${JOBNAME}/"*.pdb "${JOBNAME}/"*.cif "${JOBNAME}/structs"
  SUFFIX="custom_structs=True"
fi
snakemake --cores $(nproc --all) --use-conda -s fold_tree/workflow/fold_tree --config folder="./${JOBNAME}" filter=False $SUFFIX  #> --use-conda /dev/null 2>&1
#snakemake --cores 4 --use-conda -s fold_tree/workflow/fold_tree --config folder=./${jobname} filter=False

/content/fold_tree/


Config file /content/fold_tree/workflow/config/config_vars.yaml is extended by additional config specified via the command line.
Building DAG of jobs...
Relative file path './test_1634a/plddt.json' starts with './'. This is redundant and strongly discouraged. It can also lead to inconsistent results of the file-matching approach used by Snakemake. You can simply omit the './' for relative file paths.
Relative file path './test_1634a/foldtree_struct_tree.PP.nwk.rooted.final' starts with './'. This is redundant and strongly discouraged. It can also lead to inconsistent results of the file-matching approach used by Snakemake. You can simply omit the './' for relative file paths.
Relative file path './test_1634a/alntmscore_struct_tree.PP.nwk.rooted.final' starts with './'. This is redundant and strongly discouraged. It can also lead to inconsistent results of the file-matching approach used by Snakemake. You can simply omit the './' for relative file paths.
Relative file path './test_1634a

CalledProcessError: Command 'b'JOBNAME=$1\nINPUT_TYPE=$2\nSUFFIX=""\nif [[ $INPUT_TYPE = "custom" ]]; then\n  mkdir -p "${JOBNAME}/structs"\n  mv "${JOBNAME}/"*.pdb "${JOBNAME}/"*.cif "${JOBNAME}/structs"\n  SUFFIX="custom_structs=True"\nfi\nsnakemake --cores $(nproc --all)  -s fold_tree/workflow/fold_tree --config folder="./${JOBNAME}" filter=False $SUFFIX  #> --use-conda /dev/null 2>&1\n#snakemake --cores 4 --use-conda -s fold_tree/workflow/fold_tree --config folder=./${jobname} filter=False\n'' returned non-zero exit status 1.

In [ ]:
#@title Plot Foldtree output {run: "auto"}
tree = "foldseek_rooted" #@param ["foldseek_rooted", "foldseek", "lddt_rooted", "lddt", "alntmscore_rooted", "alntmscore"]
import sys
if f"/usr/local/lib/python{python_version}/site-packages/" not in sys.path:
    sys.path.insert(0, f"/usr/local/lib/python{python_version}/site-packages/")

import os
os.environ['QT_QPA_PLATFORM']='offscreen'
from ete3 import Tree, TreeStyle, TextFace, CircleFace

filelookup = {
    "foldseek_rooted": "foldtree_struct_tree.PP.nwk.rooted.final",
    "foldseek": "foldtree_struct_tree.PP.nwk",
    "lddt_rooted": "lddt_struct_tree.PP.nwk.rooted.final",
    "lddt": "lddt_struct_tree.PP.nwk",
    "alntmscore_rooted" : "alntmscore_struct_tree.PP.nwk.rooted.final",
    "alntmscore" :  "alntmscore_struct_tree.PP.nwk"
}

t = Tree(f"{jobname}/{filelookup[tree]}", format = 0)
# Define a tree style
ts = TreeStyle()
ts.mode = "c"  # This sets the tree layout to radial
ts.show_leaf_name = True
ts.show_branch_length = True
ts.show_branch_support = True

for n in t.traverse():
    support_face = CircleFace(radius=10, color="Thistle", style="circle")
    n.add_face(support_face, column=0, position="branch-right")
    n.img_style["vt_line_width"] = 50
    n.img_style["hz_line_width"] = 50


for leaf in t.iter_leaves():
    leaf.img_style["vt_line_type"] = 1  # for vertical lines
    leaf.img_style["hz_line_type"] = 1  # for horizontal lines
    leaf.add_face(TextFace(leaf.name, fsize=512), column=0, position="branch-right")

# Visualize the tree
t.render(jobname + "/tree.svg", w=1000, h=1000, units="px", tree_style=ts, dpi=300)

import base64
from IPython.display import display, HTML

with open(jobname + '/tree.svg', 'r') as f:
  display(HTML('<img style="width:100%; background:white; height:100%;max-width: 80vw;margin:1em;" src="data:image/svg+xml;base64,' + base64.b64encode(f.read().encode('ascii')).decode('ascii') + '" />'))


## Tree visualisation and comparison
The tree visulasition below is powered by [Phylo.io](https://beta.phylo.io/viewer/)
You can select structural distance metrics to compare tree topologies.
Comparison with sequence based trees is coming soon.

### Usage
Use the dropdown menus to select the rooted or unrooted tree, the distance metric and the tree to display.
The best results in the manuscript were obtained with the Foldseek score.

To return to a single tree view, select no tree in the second dropdown menu.
The color of the branches represents the maximum jaccard similarity between that subtree's leafset and the closest matching subtree's leafset in the tree on the opposite side of the visualization.
The darker the of the branch leading up to a node, the more similar the sets of leaves are.

In [ ]:
#@title Phyloio visualization
!cp -r /content/fold_tree/docs/dist_server/* /usr/local/share/jupyter/nbextensions/google.colab
!cp -r /content/{jobname} /usr/local/share/jupyter/nbextensions/google.colab
import csv
filelookup = {
    "foldseek_rooted": "foldtree_struct_tree.PP.nwk.rooted.final",
    "foldseek": "foldtree_struct_tree.PP.nwk",
    "lddt_rooted": "lddt_struct_tree.PP.nwk.rooted.final",
    "lddt": "lddt_struct_tree.PP.nwk",
    "alntmscore_rooted" : "alntmscore_struct_tree.PP.nwk.rooted.final",
    "alntmscore" :  "alntmscore_struct_tree.PP.nwk"
}

id_mapper = {}

with open(jobname +  '/' + 'finalset.csv', newline='') as csvfile:
    for row in csv.reader(csvfile, delimiter=','):
      id_mapper[row[3]] = row[2]

with open('fold_tree/docs/dist_server/compare_tree.html', 'r') as f:
      html_string = f.read()
      html_string = html_string.replace( u'\u200b', '' )

      for key, value in filelookup.items():

        with open(jobname + '/' + value, 'r') as f:
          output = f.read()

          for name, name_species in id_mapper.items():
            output = output.replace( name, name_species )

          html_string = html_string.replace( key + '_123456789', output )

from IPython.display import HTML
HTML(html_string)

In [ ]:
#@title Package and download results
!zip -FSr $jobname".result.zip" $jobname
files.download(f"{jobname}.result.zip")